# K-Means Clustering dengan scikit-learn pada Data Penindakan KPK 2019–2025
Notebook pendamping artikel *Penerapan Algoritma K-Means Clustering dengan Scikit-Learn pada Data Penindakan KPK untuk Pemetaan Wilayah dan Sektor Rawan Korupsi*.

**Cara pakai di Google Colab:** menu *Runtime → Run all* (atau jalankan sel satu per satu dengan Shift + Enter).
Tidak perlu mengunggah file apa pun, karena data sudah ada di dalam notebook.

Alur analisis:
1. Data → 2. Normalisasi → 3. Menentukan jumlah kelompok (k) → 4. Menjalankan K-Means → 5. Grafik hasil → 6. Ringkasan

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import davies_bouldin_score, silhouette_score

SEED = 42
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
WARNA = ["#E2703A", "#2F6FB0", "#12203A", "#B9B2A3"]   # Kelompok 1, 2, 3, 4
print("Library siap.")

## 1. Data asli KPK (per tahun)
Sumber: **kpk.go.id → Publikasi Data → Statistik → Penindakan** (per Maret 2026), tabel *TPK Berdasarkan Wilayah, Instansi, dan Jenis Perkara*.
Angka di bawah adalah jumlah perkara tahap **penyidikan** setiap tahun, 2019–2025. Sel kosong di situs KPK ditulis 0.

In [ ]:
TAHUN = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

def tabel(baris):
    return pd.DataFrame(baris, index=TAHUN).T

wilayah_tahunan = tabel({
    "Pemerintah Pusat":   [55, 29, 22, 20, 59, 39, 46],
    "Jawa":               [28, 19, 23, 48, 41, 66, 25],
    "Sumatera":           [44, 29, 43, 19, 10, 26, 31],
    "Kalimantan":         [16,  6,  5, 11,  9, 16,  6],
    "Sulawesi":           [ 3,  7,  8,  8,  7,  0,  8],
    "Papua":              [ 2,  1,  0,  9, 17,  2,  0],
    "Maluku":             [ 0,  0,  4,  5,  9,  4,  0],
    "Kep. Nusa Tenggara": [ 3,  0,  0,  0,  9,  0,  0],
    "Bali":               [ 0,  0,  3,  0,  0,  1,  0],
})
instansi_tahunan = tabel({
    "Pemkab/Pemkot":       [66, 50, 75, 65, 53, 30, 45],
    "Kementerian/Lembaga": [44, 12, 19, 26, 52, 40, 31],
    "BUMN/BUMD":           [17, 16,  8, 12, 34, 38, 28],
    "Pemerintah Provinsi": [11, 11,  5, 16, 22, 27,  7],
    "DPR RI":              [ 7,  0,  1,  1,  0, 10,  5],
    "Komisi":              [ 0,  2,  0,  0,  0,  9,  0],
})
jenis_tahunan = tabel({
    "Gratifikasi/Penyuapan":   [119, 55, 65, 100, 85, 63, 48],
    "Pengadaan Barang/Jasa":   [ 18, 27, 30,  14, 62, 68, 39],
    "Pungutan/Pemerasan":      [  1,  0,  0,   1,  1, 16, 27],
    "TPPU":                    [  5,  3,  7,   5,  8,  6,  2],
    "Penyalahgunaan Anggaran": [  2,  6,  3,   0,  0,  0,  0],
    "Perizinan":               [  0,  0,  2,   0,  3,  0,  0],
    "Merintangi Proses KPK":   [  0,  0,  1,   0,  2,  1,  0],
})
TAHUNAN = {"Wilayah": wilayah_tahunan, "Instansi": instansi_tahunan, "Jenis perkara": jenis_tahunan}
wilayah_tahunan

In [ ]:
# Grafik data mentah: jumlah perkara per tahun untuk setiap wilayah
ax = wilayah_tahunan.T.plot(figsize=(10, 4), marker="o")
ax.set_title("Perkara penyidikan KPK per tahun, menurut wilayah"); ax.set_xlabel("Tahun"); ax.set_ylabel("Perkara")
ax.legend(fontsize=7, ncol=3); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 1b. Meringkas setiap unit menjadi empat ciri
- **Total**: jumlah perkara selama tujuh tahun
- **Maks**: jumlah perkara di tahun paling ramai
- **SD**: seberapa naik-turun jumlahnya tiap tahun (simpangan baku)
- **Tren**: kemiringan garis regresi, yaitu rata-rata kenaikan (+) atau penurunan (−) per tahun

In [ ]:
CIRI = ["Total", "Maks", "SD", "Tren"]

def hitung_ciri(df):
    return pd.DataFrame({
        "Total": df.sum(axis=1),
        "Maks":  df.max(axis=1),
        "SD":    df.std(axis=1, ddof=0).round(2),
        "Tren":  df.apply(lambda r: np.polyfit(TAHUN, r.values, 1)[0], axis=1).round(2) + 0.0,
    })

DATA = {nama: hitung_ciri(df) for nama, df in TAHUNAN.items()}
# Hasilnya sama dengan Tabel 3 di artikel
DATA["Wilayah"]

## 2. Normalisasi Min-Max
Semua ciri diubah ke rentang 0–1 supaya **Total** (ratusan) tidak mendominasi **Tren** (satuan).

In [ ]:
X_NORM = {nama: pd.DataFrame(MinMaxScaler().fit_transform(df[CIRI]), index=df.index, columns=CIRI)
          for nama, df in DATA.items()}
X_NORM["Wilayah"].round(3)

## 3. Menentukan jumlah kelompok (k)
- **DBI**: makin kecil makin baik (dasar pemilihan k)
- **SSE / Elbow**: cari titik siku
- **Silhouette**: makin besar makin baik (pembanding)

In [ ]:
hasil_k = []
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for i, (nama, X) in enumerate(X_NORM.items()):
    ks = range(2, min(4, len(X) // 2) + 1)
    dbi, sse, sil = [], [], []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=50, random_state=SEED).fit(X)
        dbi.append(davies_bouldin_score(X, km.labels_)); sse.append(km.inertia_)
        sil.append(silhouette_score(X, km.labels_))
        hasil_k.append({"Data": nama, "k": k, "DBI": dbi[-1], "SSE": sse[-1], "Silhouette": sil[-1],
                        "Anggota terkecil": np.bincount(km.labels_).min()})
    ax[i].plot(ks, dbi, "o-", color=WARNA[0], label="DBI (kecil = baik)")
    ax[i].plot(ks, sil, "s-", color=WARNA[1], label="Silhouette (besar = baik)")
    ax[i].set_title(nama); ax[i].set_xlabel("k"); ax[i].set_xticks(list(ks)); ax[i].grid(alpha=.3)
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

tabel_k = pd.DataFrame(hasil_k).set_index(["Data", "k"]).round(3)
tabel_k

In [ ]:
# k terpilih = DBI terendah untuk setiap data
K_PILIH = tabel_k["DBI"].groupby(level=0).idxmin().apply(lambda x: x[1]).to_dict()
print("Jumlah kelompok terpilih (DBI terendah):", K_PILIH)

## 4. Menjalankan K-Means
Nomor kelompok diurutkan dari rata-rata **Total** terbesar, jadi *Kelompok 1* selalu yang perkaranya paling banyak.

In [ ]:
def jalankan_kmeans(nama):
    df, X = DATA[nama].copy(), X_NORM[nama]
    km = KMeans(n_clusters=K_PILIH[nama], n_init=50, random_state=SEED).fit(X)
    df["lab"] = km.labels_
    urut = df.groupby("lab")["Total"].mean().sort_values(ascending=False).index
    df["Kelompok"] = df["lab"].map({lab: i + 1 for i, lab in enumerate(urut)})
    return df.drop(columns="lab")

HASIL = {nama: jalankan_kmeans(nama) for nama in DATA}
HASIL["Wilayah"]

In [ ]:
HASIL["Instansi"]

In [ ]:
HASIL["Jenis perkara"]

## 5. Grafik hasil
Panjang batang = total perkara 2019–2025, warna = kelompok.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, (nama, df) in zip(ax, HASIL.items()):
    d = df.sort_values("Total")
    a.barh(d.index, d["Total"], color=[WARNA[k - 1] for k in d["Kelompok"]])
    for y, (tot, k) in enumerate(zip(d["Total"], d["Kelompok"])):
        a.text(tot + 5, y, f"{tot}  (K{k})", va="center", fontsize=8)
    a.set_title(f"{nama} (k = {K_PILIH[nama]})"); a.set_xlabel("Total perkara")
    a.set_xlim(0, d["Total"].max() * 1.25); a.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Peta posisi: besar perkara (Total) vs arah (Tren)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, (nama, df) in zip(ax, HASIL.items()):
    for k, g in df.groupby("Kelompok"):
        a.scatter(g["Total"], g["Tren"], s=90, color=WARNA[k - 1], label=f"Kelompok {k}", edgecolor="white")
    for n, r in df.iterrows():
        a.annotate(n, (r["Total"], r["Tren"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    a.axhline(0, color="gray", lw=.8, ls="--")
    a.set_title(nama); a.set_xlabel("Total perkara"); a.set_ylabel("Tren per tahun (+ naik, − turun)")
    a.legend(fontsize=7); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Profil dan ringkasan setiap kelompok

In [ ]:
for nama, df in HASIL.items():
    print(f"=== {nama} ===")
    for k, g in df.groupby("Kelompok"):
        arah = "naik" if g["Tren"].mean() > 1.5 else ("turun" if g["Tren"].mean() < -1.5 else "relatif stabil")
        print(f"Kelompok {k}: {', '.join(g.index)}")
        print(f"   rata-rata {g['Total'].mean():.1f} perkara, tren {g['Tren'].mean():+.2f}/tahun ({arah})")
    print()